In [1]:
import numpy as np
import pandas as pd
from src.environments.simple_trading_env import SimpleTradingEnv
from src.utils.indicator_utils import add_indicators
import json

# Load data
symbol = 'BTCUSDT'
timeframe = '5m'
data_path = f'data/binance-{symbol}-{timeframe}.pkl'
df = pd.read_pickle(data_path)

print(f'Loaded {len(df)} rows for {symbol} {timeframe}')

df['date'] = pd.to_datetime(df['date_close'])
add_indicators(df)
df = df.dropna().reset_index(drop=True)

# Use recent data for testing
total_timesteps = 1000
test_data = df.iloc[-total_timesteps:]

# Create environment (no wrappers for direct testing)
env = SimpleTradingEnv(test_data, initial_balance=10000, device="cuda")

print(f"\n{'='*70}")
print("Environment initialized - Ready for action testing")
print(f"{'='*70}")
print(f"Initial Balance: ${env.initial_balance:,.2f}")
print(f"Action Space: {env.action_space}")
print(f"  Action format: [direction, risk_reward_ratio, atr_multiplier]")
print(f"  Direction: 0=HOLD, 1=LONG, 2=SHORT, 3=CLOSE")
print(f"  Risk-Reward: 0-9 (maps to 1.0x-10.0x)")
print(f"  ATR Multiplier: 0-9 (maps to 1.5-3.3)")
print(f"{'='*70}\n")

Loaded 264323 rows for BTCUSDT 5m

Environment initialized - Ready for action testing
Initial Balance: $10,000.00
Action Space: MultiDiscrete([ 4 10 10])
  Action format: [direction, risk_reward_ratio, atr_multiplier]
  Direction: 0=HOLD, 1=LONG, 2=SHORT, 3=CLOSE
  Risk-Reward: 0-9 (maps to 1.0x-10.0x)
  ATR Multiplier: 0-9 (maps to 1.5-3.3)



## Test 1: Random Actions

In [3]:
np.random.seed(42)
obs, _ = env.reset()

steps_to_run = 100
history_records = []

print(f"Running {steps_to_run} steps with RANDOM actions...")
print(f"{'='*70}\n")

for step in range(steps_to_run):
    action = env.action_space.sample()
    obs, reward, done, truncated, info = env.step(action)
    history_records.append(info)
    if done or truncated:
        print(f"Episode ended at step {step}")
        break

df_history = pd.DataFrame(history_records)
df_history['direction'] = df_history['action'].apply(lambda x: x[0] if isinstance(x, np.ndarray) else x)

# Summary
print("RANDOM ACTION TEST SUMMARY")
print(f"{'='*70}")
print(f"Total Steps: {len(df_history)}")
print(f"Total Reward: {df_history['reward'].sum():.4f} | Avg/Step: {df_history['reward'].mean():.4f}")
print(f"Final Equity: ${df_history['equity'].iloc[-1]:,.2f}")
print(f"P&L: ${df_history['equity'].iloc[-1] - df_history['equity'].iloc[0]:,.2f}")

# Action distribution
action_names = {0: 'HOLD', 1: 'LONG', 2: 'SHORT', 3: 'CLOSE'}
action_counts = df_history['direction'].value_counts().sort_index()
print(f"\nAction Distribution:")
for action_id, count in action_counts.items():
    print(f"  {action_names.get(action_id, action_id):>6}: {count:3d} ({count/len(df_history)*100:5.1f}%)")

# Reward by action
print(f"\nReward by Action:")
reward_summary = df_history.groupby('direction')['reward'].agg(['count', 'mean', 'sum']).round(4)
for idx, row in reward_summary.iterrows():
    print(f"  {action_names.get(idx, idx):>6}: count={row['count']:3.0f}, mean={row['mean']:7.4f}, sum={row['sum']:8.4f}")

print(f"{'='*70}\n")

Running 100 steps with RANDOM actions...

RANDOM ACTION TEST SUMMARY
Total Steps: 100
Total Reward: -231.3117 | Avg/Step: -2.3131
Final Equity: $8,588.16
P&L: $-1,411.84

Action Distribution:
    HOLD:  28 ( 28.0%)
    LONG:  21 ( 21.0%)
   SHORT:  21 ( 21.0%)
   CLOSE:  30 ( 30.0%)

Reward by Action:
    HOLD: count= 28, mean= 0.0179, sum=  0.5000
    LONG: count= 21, mean=-2.4156, sum=-50.7282
   SHORT: count= 21, mean=-3.6219, sum=-76.0609
   CLOSE: count= 30, mean=-3.5008, sum=-105.0227



## Test 2: Custom action sequence

In [2]:
# Reset environment for sequence test
env.reset()

# Define specific action sequence
action_sequence = (
    [[0, 0, 0]] * 2 +      # Wait 2 steps
    [[1, 4, 4]] +          # Enter LONG (RR=4.6x, ATR=2.4)
    [[0, 0, 0]] * 20 +     # Hold for 20 steps
    [[3, 0, 0]]            # Close position
)

print(f"Sequence: HOLD(2) → LONG → HOLD(20) → CLOSE")
print(f"{'='*80}\n")

history_records = []
for step, action in enumerate(action_sequence):
    obs, reward, done, truncated, info = env.step(action)
    history_records.append(info)
    if done or truncated:
        print(f"Episode ended at step {step}")
        break

df_history = pd.DataFrame(history_records)

# Summary
print(f"SEQUENCE TEST SUMMARY")
print(f"{'='*80}")
print(f"Steps: {len(df_history)} | Total Reward: {df_history['reward'].sum():.4f}")
print(f"Initial: ${df_history['equity'].iloc[0]:,.2f} | Final: ${df_history['equity'].iloc[-1]:,.2f}")
print(f"P&L: ${df_history['equity'].iloc[-1] - df_history['equity'].iloc[0]:,.2f} ({(df_history['equity'].iloc[-1] / df_history['equity'].iloc[0] - 1) * 100:.2f}%)")
print(f"{'='*80}\n")

# Show only steps with rewards != 0 or position changes
print("RELEVANT STEPS (Rewards & Position Changes):")
print(f"{'Step':>4} | {'Action':>6} | {'Reward':>8} | {'Equity':>10} | {'UnrPnL%':>8} | {'Note'}")
print(f"{'-'*80}")

action_names = {0: 'HOLD', 1: 'LONG', 2: 'SHORT', 3: 'CLOSE'}
prev_pos = 0

for i, row in df_history.iterrows():
    action_dir = row['action'][0] if isinstance(row['action'], (list, np.ndarray)) else row['action']
    action_name = action_names.get(action_dir, str(action_dir))
    pos_size = row['position_size']
    unrealized = row.get('unrealized_pnl', 0)
    used_bal = row.get('used_balance', 0)
    unrealized_pct = (unrealized / used_bal * 100) if used_bal > 0 else 0
    
    # Show if: reward != 0, position changed, or first/last step
    show_step = (
        row['reward'] != 0 or 
        pos_size != prev_pos or 
        i == 0 or 
        i == len(df_history) - 1
    )
    
    if show_step:
        note = ""
        if pos_size > 0 and prev_pos == 0:
            note = "ENTERED POSITION"
        elif pos_size == 0 and prev_pos > 0:
            note = "CLOSED POSITION"
        elif row['reward'] > 0:
            note = "Holding reward"
        
        print(f"{row['step']:4d} | {action_name:>6} | {row['reward']:8.4f} | ${row['equity']:9.2f} | {unrealized_pct:7.2f}% | {note}")
    
    prev_pos = pos_size

print(f"{'-'*80}\n")

# Trade analysis
closed_trades = [t for t in history_records[-1].get('trades', []) if t.get('status') == 'CLOSED']
if closed_trades:
    trade = closed_trades[-1]
    print(f"TRADE ANALYSIS")
    print(f"{'='*80}")
    print(f"Direction: {'LONG' if trade['direction'] == 1 else 'SHORT'} | Duration: {trade.get('duration', 0)} steps")
    print(f"Entry: ${trade['entry_price']:,.2f} | Exit: ${trade.get('exit_price', 0):,.2f}")
    print(f"PnL: ${trade['pnl']:,.2f} ({trade['pnl_percent']*100:.2f}%) | Exit: {trade['reason']}")
    
    # Calculate actual vs expected reward
    duration = trade.get('duration', 0)
    pnl = trade['pnl']
    pnl_pct = trade['pnl_percent']
    
    # Get actual reward from close step
    close_step = next((r for r in history_records if r['position_size'] == 0 and any(t['status'] == 'CLOSED' for t in r.get('trades', []))), None)
    actual_reward = close_step['reward'] if close_step else 0
    
    # Calculate expected
    if 'TP' in trade['reason']:
        base = 50.0 + abs(pnl_pct) * 5.0
        duration_mult = 1.2 if 5 <= duration <= 100 else (0.8 if duration < 5 else 1.0)
        expected = base * duration_mult
        reward_type = f"TP (base={base:.1f}, mult={duration_mult:.1f})"
    elif 'SL' in trade['reason']:
        base = 5.0 + abs(pnl_pct) * 1.0
        base *= (0.7 if duration < 10 else 1.0)
        expected = -base
        reward_type = f"SL"
    else:
        expected = -2.0 if pnl > 0 else -1.0
        reward_type = "Manual Exit"
    
    print(f"\nREWARD v14.1: {actual_reward:.4f} | Expected: {expected:.4f} ({reward_type})")
    print(f"Status: {'✓ Correct' if abs(actual_reward - expected) < 0.01 else '✗ Mismatch'}")
    print(f"{'='*80}")

Sequence: HOLD(2) → LONG → HOLD(20) → CLOSE

SEQUENCE TEST SUMMARY
Steps: 24 | Total Reward: -5.3000
Initial: $10,000.00 | Final: $10,051.07
P&L: $51.07 (0.51%)

RELEVANT STEPS (Rewards & Position Changes):
Step | Action |   Reward |     Equity |  UnrPnL% | Note
--------------------------------------------------------------------------------
 288 |   HOLD |   0.0000 | $ 10000.00 |    0.00% | 
 290 |   LONG |  -0.0500 | $  9991.67 |    0.00% | ENTERED POSITION
 291 |   HOLD |  -0.0500 | $  9985.13 |   -0.02% | 
 292 |   HOLD |  -0.0500 | $  9968.20 |   -0.06% | 
 293 |   HOLD |  -0.0500 | $  9958.71 |   -0.08% | 
 294 |   HOLD |  -0.0500 | $  9978.96 |   -0.03% | 
 295 |   HOLD |  -0.0500 | $  9986.80 |   -0.01% | 
 296 |   HOLD |  -0.0500 | $  9961.53 |   -0.07% | 
 297 |   HOLD |  -0.0500 | $  9968.63 |   -0.06% | 
 298 |   HOLD |  -0.0500 | $  9978.96 |   -0.03% | 
 299 |   HOLD |   0.1000 | $ 10039.41 |    0.11% | Holding reward
 300 |   HOLD |   0.1000 | $ 10044.28 |    0.13% | Hol